<center><img src="https://raw.githubusercontent.com/Hamid-abdellaoui/crop-yield-predection/main/nb_bg.png"/></center>


## 1. Data Preparation

### 1.1 Standard imports and load datasets

In [ ]:
pip install pmdarima==1.8.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
path='https://raw.githubusercontent.com/Hamid-abdellaoui/crop-yield-predection/main/Daily%2030%20years.csv'
data = pd.read_csv(path, sep=';' , parse_dates = ['date'], index_col = ['date'])
data.head()

✔ tavg : Température moyenne <br>
✔ tmin : Température min<br>
✔ tmax : Température max<br>
✔ prcp : quantités précipétations en mm<br>

In [ ]:
data.shape

✔ Notre Dataset contient quatre colonne et 11324 observations

In [ ]:
data.describe().transpose()

On va supprimmer les deux colonnes tmin et tmax car on'en aura pas besoin

In [ ]:
data=data.drop(['tmax','tmin'], axis=1)
data.isna().sum()

### 1.2 les valeures manquantes

✔ Il y'a des valeurs manquantes, 31% des données de la colonne précipétations sont manquantes. <br>
✔ On va  remplisser ces valeurs manquantes de températures par interpolation linéaire.
<br> ✔ la formule d'interpolation est :<br>

<img src='https://wikimedia.org/api/rest_v1/media/math/render/svg/4953daa93f7583973d7ae5716f53873f96b42afb'/>

In [ ]:
data['prcp'] = data['prcp'].fillna(0)
data.isna().sum()

In [ ]:
data['prcp'] = data['prcp'].fillna(0)
data.isna().sum()

### 1.3 rééchantillonnage

#####          Données mensuelles

In [ ]:
monthlyData=pd.DataFrame()
monthlyData['prcp']=data['prcp'].resample('M').sum()
monthlyData['tavg']=data['tavg'].resample('M').mean()
monthlyData.drop(monthlyData.tail(1).index,inplace=True)
monthlyData.info()

#####          Données annuelles

In [ ]:
yearlyData=pd.DataFrame()
yearlyData['prcp']=monthlyData['prcp'].resample('Y').sum()
yearlyData['tavg']=monthlyData['tavg'].resample('Y').mean()
yearlyData.drop(yearlyData.tail(1).index,inplace=True)
yearlyData.info()

####   Ploting

In [ ]:
fig, ax = plt.subplots(figsize=(40, 10))

plt.subplot(121)
plt.xlabel('Date')
plt.ylabel('Précipetations',fontsize=35)
plt.plot(monthlyData[['prcp']])

plt.subplot(122)
plt.xlabel('Date')
plt.ylabel('Températures',fontsize=34)
plt.plot(monthlyData[['tavg']])

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(30, 7))
plt.subplot(121)
plt.xlabel('Date')
plt.ylabel('Précipétations annuelles',fontsize=28)
plt.plot(yearlyData['prcp'])

plt.subplot(122)
plt.xlabel('Date')
plt.ylabel('Moyennes de températures annuelles',fontsize=20)
plt.plot(yearlyData[['tavg']])

plt.show()

In [ ]:
 print(yearlyData.iloc[6])

!!  Same as in https://meteostat.net/fr/place/MA-ESIT?t=1996-01-01/1996-12-31

## 2. Temperatures

### 2.1 Importation et exploration

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima_model import ARIMA
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()
from matplotlib import pyplot
import statsmodels
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statistics

In [ ]:
temperature = monthlyData[['tavg']].copy()
temperature.info()

In [ ]:
fig, ax = plt.subplots(figsize=(30, 7))
plt.xlabel('Date')
plt.ylabel('Précipetations',fontsize=30)
plt.plot(temperature)
plt.show()

### 2.2 Stationnarité

#### Par observation

In [ ]:
fig, ax = plt.subplots(figsize=(30, 7))
rolling_mean = temperature.rolling(window = 12).mean()
plt.plot(temperature, color = 'blue', label = 'Origine')
plt.plot(rolling_mean, color = 'red', label = 'Moyenne mobile')
plt.legend(loc = 'best')
plt.title('Moyenne et serie originale')
plt.show()

✔ Le  moyen mobile est quasi- constante donc on peut estimer que notre série de données est stationnaires

#### Par test de Dickey Fuller

In [ ]:
result = adfuller(temperature['tavg'])

print('Statistiques ADF : {}'.format(result[0]))
print('p-value : {}'.format(result[1]))
print('Valeurs Critiques :')
for key, value in result[4].items():
    print('\t{}: {}'.format(key, value))

✔ La p-value est inférieure au seuil de 0,05 et la statistique ADF est proche des valeurs critiques. Par conséquent, la série temporelle est bel et bien stationnaire.<br>

### 2.3 Application du modèle SARIMA (Seasonal Autoregressive Integrated Moving Averages)

#### Importation du package

In [ ]:
#import pmdarima

In [ ]:
from pmdarima import auto_arima
# Ignore harmless warnings
import warnings
warnings.filterwarnings("ignore")

#### 2.3.1 Saisonalité

✔ Le modèle est additif , on procède alors à la décompsition pour observer la saisonalité

In [ ]:
result = seasonal_decompose(temperature['tavg'],model='add')
fig, ax =plt.subplots(figsize=(17, 4))
plt.plot(result.trend, label='trend');
plt.plot(result.seasonal, label='seasonal');
plt.plot(result.resid, label='residual');
fig.legend(fontsize=15)

<font size=4>✔ y(t)  =  Trend(t) + Seasonality(t) + Residual(t) </font> <br>
✔ d'après la decompisition on constate clairement la présence d'une saisonalité

In [ ]:
fig, ax =plt.subplots(figsize=(17, 4))

result.seasonal.iloc[0:24].plot()

✔ une saisonalité d'une année = 12 mois
<br>
✔ On doit donc choisir le modèle SARIMA qui est compatibe avec ce type de séries

#### 2.3.2 On cherche les ordres du modèle SARIMA

In [ ]:
auto_arima(temperature['tavg'], seasonal=True, m=12).summary()

* Intercep : the constant term <br>
* The ar.L1, ma.L2.... and ar.S.L12 are our lag variables <br>
* Sigma2 : error term <br>
* Sample : range of data <br>
* les termes significants ont une p-Value < 0.05 (we want each term to have a p-value of less than 0.05, so we can reject the null hypothesis with values that ARE statistically significant) <br>

* Ljung-Box : tester les erreurs s'ils sont des bruits blancs(here Since the probability is above 0.05, we can't reject the null that the errors are white noise.)

* Jarque-Bera tests for the normality of errors. It tests the null that the data is normally distributed against an alternative of another distribution


* Heteroscedasticity tests that the error residuals are homoscedastic or have the same variance

* Log-Likelihood, AIC, BIC, et HQIC sond des critères de performance

```
 https://analyzingalpha.com/blog/interpret-arima-results

```



✔ Donc on va modéliser cette série par un modèle SARIMA

### 2.4 Spliting data for testing

In [ ]:
fig, ax = plt.subplots(figsize=(30, 10))
# Set one year for testing
train = temperature.iloc[:361]
valid = temperature.iloc[360:]
#preprocessing (since arima takes univariate series as input)
#train.drop('date',axis=1,inplace=True)
#valid.drop('date',axis=1,inplace=True)

#plotting the data
plt.plot(train['tavg'], color = 'red', label='train data')
plt.plot(valid['tavg'], color = 'green', label ='test data')
fig.legend(fontsize=30)
plt. show()

In [ ]:
train.head()

#### Fiting the model

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
model = SARIMAX(train['tavg'],order=(2, 0, 3),seasonal_order=(1, 0, 2, 12))
results = model.fit()
results.summary()

#### Testing

In [ ]:
start=len(train)-1
end=len(train)+len(valid)-2
predictions = results.predict(start=start, end=end, dynamic=False, typ='levels').rename('SARIMA(1,0,3)(1,0,2,12) Predictions')

In [ ]:
predictions.to_frame().info()

In [ ]:
for i in range(0,len(predictions)):
    print('Value =', round (predictions[i],3),'  ;  Expected =', round(valid.values[i][0],3),'      Expected- value =', round(-predictions[i]+valid.values[i][0],2))

#### Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error

<b>✔ </b> Two calculations related to linear regression are <a href='https://en.wikipedia.org/wiki/Mean_squared_error'><strong>mean squared error</strong></a> (MSE) and <a href='https://en.wikipedia.org/wiki/Root-mean-square_deviation'><strong>root mean squared error</strong></a> (RMSE)

<b>✔ </b>The formula for the mean squared error is<br>
&nbsp;&nbsp;&nbsp;&nbsp;$MSE = {\frac 1 L} \sum\limits_{l=1}^L (y_{T+l} - \hat y_{T+l})^2$<br><br>
where $T$ is the last observation period and $l$ is the lag point up to $L$ number of test observations.

<b>✔ </b>The formula for the root mean squared error is<br>
&nbsp;&nbsp;&nbsp;&nbsp;$RMSE = \sqrt{MSE} = \sqrt{{\frac 1 L} \sum\limits_{l=1}^L (y_{T+l} - \hat y_{T+l})^2}$<br><br>

<b>✔ </b>The advantage of the RMSE is that it is expressed in the same units as the data.<br>

In [ ]:
error = mean_squared_error(valid['tavg'],predictions)
print('MSE error =', error)

In [ ]:
from statsmodels.tools.eval_measures import rmse

error = rmse(valid['tavg'], predictions)
print('RMSE Error: ', error)

✔ Des erreures RMSE et MSE faibles donc notre modèle est fiable

#### On compare les prédictions avec lesdonnées oroginelles

In [ ]:
title = 'Monthly temperature'
ylabel='in celicius'
xlabel=''

ax = valid['tavg'].plot(legend=True,figsize=(20,8),title=title)
predictions.plot(legend=True)
ax.autoscale(axis='x',tight=True)
ax.set(xlabel=xlabel, ylabel=ylabel);
plt.legend(fontsize=18)

### 2.5 Retrain the model on the full data, and forecast the future

In [ ]:
model = SARIMAX(temperature['tavg'],order=(1, 0, 3),seasonal_order=(1, 0, 2, 12))
results = model.fit()
fcast = results.predict(len(temperature),len(temperature)+11,typ='levels').rename('SARIMA(0,1,3)(1,0,1,12) Forecast')

In [ ]:
# Plot predictions against known values
title = 'Temperature mensuelles'
ylabel='en celicius'
xlabel=''

ax = temperature['tavg'].plot(legend=True,figsize=(40,14),title=title)
fcast.plot(legend=True,fontsize=20)
ax.autoscale(axis='x',tight=True)
plt.legend(fontsize=26)
ax.set(xlabel=xlabel, ylabel=ylabel);

✔  Et voici nous prédictions de l'année 2021 :

In [ ]:
fcast.head(12)

## 3. Précipétations

### 3.1  Exploring

In [ ]:
precipetation = monthlyData[['prcp']].copy()
precipetation

In [ ]:
fig, ax = plt.subplots(figsize=(20, 7))
plt.xlabel('Date')
plt.ylabel('Précipetations',fontsize=30)
plt.plot(precipetation[:])
plt.show()

#### Stationnarité

In [ ]:
result = adfuller(precipetation['prcp'])

print('Statistiques ADF : {}'.format(result[0]))
print('p-value : {}'.format(result[1]))
print('Valeurs Critiques :')
for key, value in result[4].items():
    print('\t{}: {}'.format(key, value))

<b>✔ </b> le test de Dickey fuller confirme que la série est stationnaire

#### Saisonalité

In [ ]:
result = seasonal_decompose(precipetation['prcp'],model='add')
result.plot();

✔ D'après la composante saisonière on peut supposer qu'il y'a saisonalité <br>
❌❗ Mais attention la composante <b>Residual</b> prend des valeurs très grandes( plus grandes que la <b>Trend</b> et <b>Seasonal</b> ensemble)
<br><b>➡</b> Donc la série ne présente pas une saisonnalité, on utulise donc ARIMA

In [ ]:
model = auto_arima(precipetation.prcp,test='adf',seasonal=False)

model.summary()

On va utuliser ARMA(2,0) qui est aussi ARIMA(2,0,0)

### 3.2  ARIMA

#### Spliting data for testing

In [ ]:
train = precipetation.iloc[:361]
valid = precipetation.iloc[360:]
fig, ax = plt.subplots(figsize=(30, 10))

#plotting the data
plt.plot(train['prcp'], color = 'red', label='train data')
plt.plot(valid['prcp'], color = 'green', label ='test data')
fig.legend(fontsize=30)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt. show()

In [ ]:
# Forecast
n_periods = 12
forecast, confint = model.predict(n_periods=n_periods, return_conf_int=True)
index_of_forecast = np.arange(len(precipetation.prcp), len(precipetation.prcp)+n_periods)

In [ ]:
for i in range(0,len(forecast)):
    print('Value =', round (forecast[i],3),'  ;  Expected =', round(valid.values[i][0],3) )

In [ ]:
prcp_series =pd.Series(precipetation['prcp'].values ,index=np.arange(0, len(precipetation.prcp+13)))

In [ ]:
# make series for plotting purpose
fc_series = pd.Series(forecast, index=index_of_forecast)
lower_series = pd.Series(confint[:, 0], index=index_of_forecast)
upper_series = pd.Series(confint[:, 1], index=index_of_forecast)
valid_series =pd.Series(valid['prcp'].values ,index=index_of_forecast)

# Plot
fig, ax = plt.subplots(figsize=(30, 10))
#plt.plot(prcp_series)
plt.plot(valid_series,label='TEST DATA')
plt.plot(fc_series, color='darkgreen',label='PREDECTION')
plt.fill_between(lower_series.index,
                 lower_series,
                 upper_series,
                 color='k', alpha=.15)

fig.legend(fontsize=25)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.title("les valeurs originales + prédites+ Intervalle de confiance",fontsize=35)
plt.show()

In [ ]:
# error = rmse(valid['prcp'],forecast)
# print('RMSE Error: ', error)

from sklearn.metrics import mean_squared_error
from math import sqrt
rmse=np.sqrt(mean_squared_error(valid['prcp'],forecast))
print(rmse)


#### Just preparing data for next step

In [ ]:
daily_prcp = data[['prcp']].copy()
daily_prcp.head()

In [ ]:
daily_prcp=daily_prcp[(daily_prcp.prcp!=270) & (daily_prcp.prcp!=199.9)]
daily_prcp.max()

In [ ]:
monthly=pd.DataFrame()
monthly['prcp']=daily_prcp['prcp'].resample('M').sum()
monthly.info()

In [ ]:
ax, fig=plt.subplots(figsize=(50,15))
plt.plot(monthly)

In [ ]:
monthly.to_csv('prcp.csv')

### 3.3  LSTM

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('/content/prcp.csv',sep=',', parse_dates = ['date'], index_col = ['date'])

In [ ]:
len(df)

In [ ]:
train = df.iloc[:361]
test = df.iloc[361:]
print(train.shape)
print(test.shape)

##### Scale Data

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

In [ ]:
scaler.fit(train)
scaled_train = scaler.transform(train)
scaled_test = scaler.transform(test)

In [ ]:
from keras.preprocessing.sequence import TimeseriesGenerator

In [ ]:
n_input = 12
n_features=1
generator = TimeseriesGenerator(scaled_train, scaled_train, length=n_input, batch_size=1)

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM

In [ ]:
# define model
model = Sequential()
model.add(LSTM(100 , input_shape=(n_input, n_features)))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mean_squared_error')

model.summary()

In [ ]:
# fit model
model.fit(generator,epochs=400)

In [ ]:
loss_per_epoch = model.history.history['loss']
plt.plot(range(len(loss_per_epoch)),loss_per_epoch)

In [ ]:
last_train_batch = scaled_train[-12:]

In [ ]:
last_train_batch = last_train_batch.reshape((1, n_input, n_features))

In [ ]:
model.predict(last_train_batch)

In [ ]:
scaled_test[0]

In [ ]:
test_predictions = []
import numpy as np

first_eval_batch = scaled_train[-n_input:]
current_batch = first_eval_batch.reshape((1, n_input, n_features))

for i in range(len(test)):

    # get the prediction value for the first batch
    current_pred = model.predict(current_batch)[0]

    # append the prediction into the array
    test_predictions.append(current_pred)

    # use the prediction to update the batch and remove the first value
    current_batch = np.append(current_batch[:,1:,:],[[current_pred]],axis=1)

In [ ]:
true_predictions = scaler.inverse_transform(test_predictions)

In [ ]:
test['Predictions'] = true_predictions
test

In [ ]:
test.plot(figsize=(28,5))

In [ ]:
from sklearn.metrics import mean_squared_error
from math import sqrt
rmse=np.sqrt(mean_squared_error(test['prcp'],test['Predictions']))
print(rmse)


In [ ]:
scaler.fit(df)
scaled= scaler.transform(df)
generator = TimeseriesGenerator(scaled, scaled, length=n_input, batch_size=1)
model.fit(generator,epochs=400)

In [ ]:
predictions = []

first_eval_batch = scaled_train[-n_input:]
current_batch = first_eval_batch.reshape((1, n_input, n_features))

for i in range(12):

    # get the prediction value for the first batch
    current_pred = model.predict(current_batch)[0]

    # append the prediction into the array
    predictions.append(current_pred)

    # use the prediction to update the batch and remove the first value
    current_batch = np.append(current_batch[:,1:,:],[[current_pred]],axis=1)

In [ ]:
true_predictions = scaler.inverse_transform(predictions)
true_predictions

In [ ]:
ax, fig=plt.subplots(figsize=(18,6))
plt.plot(true_predictions)

## 4. Production de blé tendre

### 4.1 Vector autoregression

In [ ]:
#import required packages
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error

In [ ]:
path="https://raw.githubusercontent.com/Hamid-abdellaoui/crop-yield-predection/main/bt.csv"
df = pd.read_csv(path, sep=';', index_col = ['date'])

df.info()

In [ ]:
path='https://raw.githubusercontent.com/Hamid-abdellaoui/crop-yield-predection/main/Daily%2030%20years.csv'
data = pd.read_csv(path, sep=';' , parse_dates = ['date'], index_col = ['date'])
data=data.drop(['tmax','tmin'], axis=1)
data['prcp'] = data['prcp'].fillna(0)
data.interpolate(method ='linear', axis=0, inplace=True)

In [ ]:
monthlyData=pd.DataFrame()
monthlyData['prcp']=data['prcp'].resample('M').sum()
monthlyData['tavg']=data['tavg'].resample('M').mean()
monthlyData.drop(monthlyData.tail(1).index,inplace=True)
monthlyData.info()

In [ ]:
monthlyData.loc[monthlyData.index.month == 5 , ['prcp']] = 0
monthlyData.loc[monthlyData.index.month == 6 , ['prcp']] = 0
monthlyData.loc[monthlyData.index.month == 7 , ['prcp']] = 0
monthlyData.loc[monthlyData.index.month == 8 , ['prcp']] = 0
monthlyData.loc[monthlyData.index.month == 9 , ['prcp']] = 0

In [ ]:
yearlyData=pd.DataFrame()
yearlyData['prcp']=monthlyData['prcp'][5:365].resample('12M').sum()
yearlyData['tavg']=monthlyData['tavg'][5:365].resample('12M').mean()
yearlyData.drop(yearlyData.head(1).index,inplace=True)
yearlyData.info()

In [ ]:
df.index=yearlyData.index
df['precipetations']=yearlyData.prcp
df['temperatures']=yearlyData.tavg
df['ratio'] = df['production']/df['superficie']
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(30, 7))
plt.subplot(121)
plt.xlabel('Date')
plt.ylabel('production+ superficie',fontsize=28)
plt.plot(df['production'])
plt.plot(df['superficie'])

plt.subplot(122)
plt.xlabel('Date')
plt.ylabel('production + precipetations',fontsize=20)
plt.plot(df['production']/1000, label='prod')
plt.plot(df['precipetations'], label='prcp')
plt.legend(fontsize=18)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
plt.xlabel('Date')
plt.ylabel('prod + Temperature',fontsize=20)
plt.plot(df['production']/9000)
plt.plot(df['temperatures'])

In [ ]:
# VAR example
from statsmodels.tsa.vector_ar.var_model import VAR
from random import random
# contrived dataset with dependency
data = list()
for i in range(30):
  v0 = df.index[i].year
  v1 = df['precipetations'][i]
  v2 = df['production'][i]
  row = [v0, v1, v2]
  data.append(row)
data

In [ ]:

data = pd.DataFrame (data, columns = ['year','prcp','prod'])
data.head()

In [ ]:
model = VAR(data)
for i in [1,2,3,4,5,6,7,8]:
    result = model.fit(i)
    print('Lag Order =', i)
    print('AIC : ', result.aic)
    print('BIC : ', result.bic)
    print('FPE : ', result.fpe)
    print('HQIC: ', result.hqic, '\n')

In [ ]:
x = model.select_order(maxlags=6)
x.summary()

In [ ]:
model_fitted = model.fit(4)
model_fitted.summary()

In [ ]:
lag_order = model_fitted.k_ar
print(lag_order)  #> 4

# Input data for forecasting
forecast_input = data.values[-lag_order:]
forecast_input

In [ ]:
# Forecast
nobs=1
fc = model_fitted.forecast(y=forecast_input, steps=nobs)
df_forecast = pd.DataFrame(fc, index=df.index[-nobs:], columns=data.columns)
df_forecast

### 4.2 Multiple linear regression

In [ ]:
pd.DataFrame(df[['precipetations','production']][:21]).corr()

les  paramètres precipetations et production sont correlés (paramètre precipetations un peu près de 63% du variable production)

In [ ]:
X = df[['precipetations', 'temperatures']][:21]
y = df['production'][:21]

In [ ]:
from sklearn import linear_model

In [ ]:
regr = linear_model.LinearRegression(normalize=False)
regr.fit(X, y)

In [ ]:
#testing (eg: 2010)
predicted = regr.predict([[387.5, 17.8]])
print(predicted[0])

la vrai valeur est : 238312.035 tonnes

In [ ]:
# les predictions pour 2021
predicted = regr.predict([[329.5, 17]])
print(predicted[0])

* Data from [Metestat](meteostat.net). <br>
* For references and more details are [here](https://github.com/Hamid-abdellaoui/Crop-yield-predection).

<!--
## BONUS: how to train a model from A to Z
<img src="https://raw.githubusercontent.com/Hamid-abdellaoui/Crop-yield-prediction/main/Comment%20trainer%20un%20mod%C3%A8le%20de%20A%20a%20Z.png"> -->


#  <center> **END**  </center>

Next two codes are used to validate the previous results

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests
maxlag=8
test = 'ssr_chi2test'
def grangers_causation_matrix(data, variables, test='ssr_chi2test', verbose=False):
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table
    are the P-Values. P-Values lesser than the significance level (0.05), implies
    the Null Hypothesis that the coefficients of the corresponding past values is
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """
    df = pd.DataFrame(np.zeros((len(variables), len(variables))), columns=variables, index=variables)
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(data[[r, c]], maxlag=maxlag)
            p_values = [round(test_result[i+1][0][test][1],4) for i in range(maxlag)]
            if verbose: print(f'Y = {r}, X = {c}, P Values = {p_values}')
            min_p_value = np.min(p_values)
            df.loc[r, c] = min_p_value
    df.columns = [var + '_x' for var in variables]
    df.index = [var + '_y' for var in variables]
    return df


grangers_causation_matrix(data, variables = ['prcp','prod'])

In [ ]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

def cointegration_test(df, alpha=0.05):
    """Perform Johanson's Cointegration Test and Report Summary"""
    out = coint_johansen(df,-1,5)
    d = {'0.90':0, '0.95':1, '0.99':2}
    traces = out.lr1
    cvts = out.cvt[:, d[str(1-alpha)]]
    def adjust(val, length= 6): return str(val).ljust(length)

    # Summary
    print('Name   ::  Test Stat > C(95%)    =>   Signif  \n', '--'*20)
    for col, trace, cvt in zip(df.columns, traces, cvts):
        print(adjust(col), ':: ', adjust(round(trace,2), 9), ">", adjust(cvt, 8), ' =>  ' , trace > cvt)

cointegration_test(data)

malheureusement the results have a weak significance !